# Time Series Analysis: Complete Guide

## 1. Overview and Key Points
- Time series analysis examines data collected over regular intervals.
- Common examples include daily stock prices, monthly sales figures, and annual rainfall totals.
- The primary objective is to identify temporal patterns and project future values.

### Key Characteristics
- Data collection occurs at fixed time points.
- Goal is to detect trends and cyclical changes.
- Application supports informed decision-making based on historical data.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Set display options for better readability
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)

print('Libraries successfully imported! Environment is ready.')

## 3. Data Creation: Synthetic Retail Sales

Time Series is defined as a sequence of data points indexed in time order. This structure allows researchers to observe how a variable changes over a specific duration.

To understand time series analysis, we need a realistic dataset. We will simulate Monthly Retail Sales spanning 10 years (120 months).

We will deliberately construct this data with:
1. A general upward Trend.
2. A recurring Seasonal pattern (higher sales in summer and winter).
3. Random noise (Residuals).

In [ ]:
# Generate temporal index (120 months spanning 10 years)
date_rng = pd.date_range(start='2015-01-01', periods=120, freq='ME')

# Time array 0 to 119
time = np.arange(120)

# 1. Trend Component (Linear upward growth)
trend = 0.5 * time

# 2. Seasonal Component (Sine wave with a 12-month period)
seasonal = 15 * np.sin(2 * np.pi * time / 12)

# 3. Residual Component (Random noise)
residual = np.random.normal(loc=0, scale=3, size=120)

# Combine components to form the final Time Series
# Base sales of 100
sales = 100 + trend + seasonal + residual

# Create DataFrame
df_retail = pd.DataFrame({
    'Date': date_rng,
    'Sales': sales,
    'Trend_True': 100 + trend,
    'Seasonal_True': seasonal,
    'Residual_True': residual
})

# Set Date as the index
df_retail.set_index('Date', inplace=True)

print('Synthetic time series dataset created with 120 months of retail data.')

## 4. Initial Data Exploration

Data preparation involves cleaning the dataset and handling missing values. It ensures the sequence is continuous and ready for mathematical treatment.

Let's preview our synthetic data.

In [ ]:
print('--- First 5 rows of Retail Sales Data ---')
print(df_retail[['Sales']].head().round(2))
print('\n--- Dataset Information ---')
df_retail[['Sales']].info()
print('\n--- Descriptive Statistics ---')
print(df_retail[['Sales']].describe().round(2))

## 5. Visualizing the Raw Time Series

Before applying any modeling, it is crucial to plot the series. Humans are excellent at spotting visual patterns.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_retail.index, df_retail['Sales'], color='blue', linewidth=2, label='Monthly Sales')
plt.title('Monthly Retail Sales (2015-2024)', fontsize=15)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Sales Volume', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Core Concept 1: Components of Time Series

A time series can be broken down into three distinct parts:

- **Trend Component**: Reflects the long-term progression of the series. Indicates whether values generally increase or decrease over time.
- **Seasonal Component**: Captures repeating patterns at fixed intervals (e.g., spikes in retail sales every December).
- **Residual Component**: Represents random noise or irregular movements. What remains after removing trend and seasonal factors.

In [ ]:
# Visualizing the underlying true components we generated
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(df_retail.index, df_retail['Sales'], color='blue')
axes[0].set_title('Original Time Series')
axes[0].set_ylabel('Sales')

axes[1].plot(df_retail.index, df_retail['Trend_True'], color='orange')
axes[1].set_title('Underlying Trend')
axes[1].set_ylabel('Trend')

axes[2].plot(df_retail.index, df_retail['Seasonal_True'], color='green')
axes[2].set_title('Seasonal Pattern')
axes[2].set_ylabel('Seasonal')

axes[3].plot(df_retail.index, df_retail['Residual_True'], color='red', marker='o', linestyle='None', markersize=3)
axes[3].set_title('Random Noise (Residuals)')
axes[3].set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 7. Core Concept 2: Decomposition

Decomposition is the mathematical process of isolating a series into its distinct parts. 
The additive model expresses the series Y_t as:

Y_t = T_t + S_t + R_t

Where T_t represents the trend, S_t the seasonal factor, and R_t the residual.
In the real world, we don't know the true components. We must estimate them using statistical tools.

In [ ]:
# Using statsmodels to decompose the time series automatically
# We use additive model and a period of 12 (monthly data)
decomposition = seasonal_decompose(df_retail['Sales'], model='additive', period=12)

df_retail['Estimated_Trend'] = decomposition.trend
df_retail['Estimated_Seasonal'] = decomposition.seasonal
df_retail['Estimated_Residual'] = decomposition.resid

print('Decomposition successful. Estimated components added to DataFrame.')

## 8. Visualizing Automated Decomposition

Let's see how well the statistical algorithm recovered the patterns we injected into the synthetic data.

In [ ]:
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.axes[0].set_title('Statsmodels Seasonal Decomposition', fontsize=14)
plt.tight_layout()
plt.show()

print('Notice the Trend starts and ends with NaN values. This is due to the moving average window used internally.')

## 9. Core Concept 3: Smoothing Methods

Smoothing methods are statistical techniques used to reduce short-term fluctuations, revealing longer-term patterns.

### Simple Moving Average (SMA)
The mathematical representation of a simple moving average is given by:
y_hat_{t+1} = (1/k) * SUM(y_{t-i}) from i=0 to k-1

Where k is the number of periods used for the average.

In [ ]:
# Calculate 6-month and 12-month Simple Moving Averages
df_retail['SMA_6'] = df_retail['Sales'].rolling(window=6).mean()
df_retail['SMA_12'] = df_retail['Sales'].rolling(window=12).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_retail.index, df_retail['Sales'], color='lightgrey', label='Original Sales')
plt.plot(df_retail.index, df_retail['SMA_6'], color='blue', label='6-Month SMA')
plt.plot(df_retail.index, df_retail['SMA_12'], color='red', label='12-Month SMA', linewidth=2)
plt.title('Simple Moving Averages', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Core Concept 4: Exponential Smoothing

Exponential smoothing provides more weight to recent observations.
It is defined as:
y_hat_{t+1} = alpha * y_t + (1 - alpha) * y_hat_t

Where alpha is a smoothing constant between 0 and 1. A higher alpha reacts faster to recent changes.

In [ ]:
# Calculate Exponential Moving Averages (EMA) with different alpha values
# ewm(alpha=...) applies the exponential weighting
df_retail['EMA_alpha_0_2'] = df_retail['Sales'].ewm(alpha=0.2, adjust=False).mean()
df_retail['EMA_alpha_0_8'] = df_retail['Sales'].ewm(alpha=0.8, adjust=False).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_retail.index, df_retail['Sales'], color='lightgrey', label='Original Sales')
plt.plot(df_retail.index, df_retail['EMA_alpha_0_2'], color='green', label='EMA (alpha = 0.2)')
plt.plot(df_retail.index, df_retail['EMA_alpha_0_8'], color='purple', label='EMA (alpha = 0.8)')
plt.title('Exponential Smoothing Comparison', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.show()

print('Observe how alpha=0.8 hugs the original line closely, tracking noise.')
print('Conversely, alpha=0.2 is much smoother, representing the trend better.')

## 11. Common Mistakes: Ignoring Stationarity

A stationary time series has statistical properties (mean, variance) that are constant over time.
Ignoring the stationarity of the data leads to incorrect model selection. 

We can test for stationarity using the Augmented Dickey-Fuller (ADF) test. 
Null Hypothesis: The series is NOT stationary.

In [ ]:
def check_stationarity(timeseries, title):
    print(f'--- Augmented Dickey-Fuller Test: {title} ---')
    result = adfuller(timeseries.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4e}')
    
    if result[1] <= 0.05:
        print('Conclusion: Reject Null Hypothesis. The data is STATIONARY.')
    else:
        print('Conclusion: Fail to Reject Null. The data is NON-STATIONARY.\n')

# Test our raw sales data
check_stationarity(df_retail['Sales'], 'Original Sales Data')

## 12. Fixing Non-Stationarity (Differencing)

Because our retail data has a clear upward trend, it is non-stationary.
We can usually fix this by taking the first difference: Y_t - Y_{t-1}.

In [ ]:
# Calculate first difference
df_retail['Sales_Diff'] = df_retail['Sales'].diff()

# Plot the differenced series
plt.figure(figsize=(12, 4))
plt.plot(df_retail.index, df_retail['Sales_Diff'], color='brown')
plt.title('Differenced Sales Series', fontsize=14)
plt.axhline(0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

# Test stationarity again
check_stationarity(df_retail['Sales_Diff'], 'Differenced Sales Data')

## 13. Common Mistakes: The Impact of External Shocks

Failing to account for external shocks that disrupt historical trends is a major pitfall.
Let's simulate a massive external shock (e.g., a supply chain crisis) and see how moving averages respond.

In [ ]:
df_shock = df_retail[['Sales']].copy()
# Inject a sudden drop in month 60
df_shock.iloc[60:63, 0] = df_shock.iloc[60:63, 0] - 60 

df_shock['SMA_12'] = df_shock['Sales'].rolling(12).mean()
df_shock['EMA'] = df_shock['Sales'].ewm(alpha=0.3).mean()

plt.figure(figsize=(12, 5))
plt.plot(df_shock.index, df_shock['Sales'], label='Sales with Shock', color='black')
plt.plot(df_shock.index, df_shock['SMA_12'], label='12-Month SMA', color='red')
plt.plot(df_shock.index, df_shock['EMA'], label='EMA (alpha=0.3)', color='blue')
plt.title('Impact of External Shocks on Smoothing', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

print('Notice how the EMA recovers and adapts much faster to the shock, while the SMA drags the shock effect out for 12 months.')

## 14. Practice Exercise: Financial Analyst Scenario

Financial analysts use daily stock prices to assess volatility and expected returns.
Let's create a synthetic daily stock price dataset using a random walk with drift.

In [ ]:
# Generate 200 days of stock data
days = 200
dates_stock = pd.date_range(start='2026-01-01', periods=days, freq='B') # Business days

# Random walk with positive drift
returns = np.random.normal(loc=0.05, scale=1.5, size=days)
stock_price = 100 + np.cumsum(returns)

df_stock = pd.DataFrame({'Price': stock_price}, index=dates_stock)
print('Synthetic Daily Stock Prices Generated.')
print(df_stock.head())

### Exercise 1: Technical Indicators
Calculate a Fast Moving Average (10-day SMA) and a Slow Moving Average (50-day SMA) for the stock prices.
Plot the original price along with both averages to look for 'crossover' points (a common trading signal).

In [ ]:
# --- EXERCISE 1 SOLUTION ---
df_stock['SMA_10'] = df_stock['Price'].rolling(window=10).mean()
df_stock['SMA_50'] = df_stock['Price'].rolling(window=50).mean()

plt.figure(figsize=(12, 6))
plt.plot(df_stock.index, df_stock['Price'], color='black', alpha=0.5, label='Daily Price')
plt.plot(df_stock.index, df_stock['SMA_10'], color='green', label='Fast SMA (10-day)')
plt.plot(df_stock.index, df_stock['SMA_50'], color='red', label='Slow SMA (50-day)')
plt.title('Stock Price Moving Average Crossover', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 2: Volatility Assessment
Calculate a 14-day rolling standard deviation (volatility) of the stock price and plot it.
This helps analysts measure risk.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
df_stock['Volatility_14'] = df_stock['Price'].rolling(window=14).std()

plt.figure(figsize=(12, 4))
plt.plot(df_stock.index, df_stock['Volatility_14'], color='purple', label='14-Day Rolling Std Dev')
plt.title('Stock Price Volatility Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Standard Deviation')
plt.legend()
plt.fill_between(df_stock.index, df_stock['Volatility_14'], color='purple', alpha=0.1)
plt.tight_layout()
plt.show()

## 15. Visualization Gallery

To summarize the impact of different smoothing constants in exponential smoothing, let's create a gallery plot comparing extreme and moderate alpha values on our retail data.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
alphas = [0.1, 0.5, 0.9]
colors = ['teal', 'darkorange', 'firebrick']

for i, alpha in enumerate(alphas):
    smoothed = df_retail['Sales'].ewm(alpha=alpha, adjust=False).mean()
    axes[i].plot(df_retail.index, df_retail['Sales'], color='lightgrey', label='Original')
    axes[i].plot(df_retail.index, smoothed, color=colors[i], linewidth=2, label=f'EMA (alpha={alpha})')
    axes[i].set_title(f'Exponential Smoothing with Alpha = {alpha}')
    axes[i].legend(loc='upper left')
    axes[i].grid(True, alpha=0.3)

plt.xlabel('Year', fontsize=12)
plt.tight_layout()
plt.show()

## 16. Summary and Key Takeaways

- **Time Series Basics**: Data indexed in time order. Key goal is identifying trends and projecting futures.
- **Core Components**: Y_t = T_t (Trend) + S_t (Seasonality) + R_t (Residual).
- **Smoothing**: Moving averages and exponential smoothing isolate true patterns from noise.
- **Stationarity**: A crucial requirement for advanced forecasting models. Checked using ADF test and fixed via differencing.
- **Common Mistakes**:
  - Overfitting models to noise rather than identifying genuine structural patterns.
  - Ignoring the stationarity of the data.
  - Failing to account for external shocks.

In [ ]:
print('Time Series Analysis Overview Module Completed.')
print('Remember: Past behavior informs future outcomes, provided we model the right signals!')